# 06 — Model Comparison

**Goal of this notebook**: put the custom CNN baseline (notebook 04) and DenseNet121
transfer learning (notebook 05) side by side — accuracy, per-class F1, and confusion
matrices — and draw a clear conclusion about which to use going forward (and put in
the Streamlit dashboard).

**Intent**: this notebook doesn't retrain anything — it loads the saved checkpoints
and CSV training logs from `models/saved_models/` and compares results that already
exist, so it stays fast to re-run.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import build_tf_dataset
from src.evaluate import evaluate_predictions

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)
CHECKPOINT_DIR = Path("../models/saved_models")

## Load both trained models

In [ ]:
custom_cnn = tf.keras.models.load_model(CHECKPOINT_DIR / "custom_cnn_baseline_best.keras")
densenet = tf.keras.models.load_model(CHECKPOINT_DIR / "densenet121_phase2_finetuned_best.keras")

## Training curves side by side

**Intent**: visually compare how each model's validation accuracy evolved — this is
the clearest way to *see* the difference transfer learning makes, not just read final
numbers.

In [ ]:
custom_cnn_history = pd.read_csv(CHECKPOINT_DIR / "custom_cnn_baseline_history.csv")
densenet_history = pd.read_csv(CHECKPOINT_DIR / "densenet121_phase2_finetuned_history.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(custom_cnn_history["val_accuracy"], label="Custom CNN (from scratch)")
ax.plot(densenet_history["val_accuracy"], label="DenseNet121 (transfer, fine-tuned)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_title("Validation accuracy: custom CNN vs. transfer learning")
ax.legend()
plt.show()

## Test-set evaluation for both models

In [ ]:
split_metadata = pd.read_csv("../data/processed/metadata_split.csv")

test_dataset_1ch = build_tf_dataset(
    split_metadata, "test", CLASS_NAMES, img_size=IMG_SIZE, batch_size=32, shuffle=False
)
test_dataset_3ch = test_dataset_1ch.map(lambda x, y: (tf.repeat(x, 3, axis=-1), y))


def get_predictions(model, dataset):
    y_true_idx, y_pred_idx = [], []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_pred_idx.extend(preds.argmax(axis=1))
        y_true_idx.extend(labels.numpy())
    y_true = np.array([CLASS_NAMES[i] for i in y_true_idx])
    y_pred = np.array([CLASS_NAMES[i] for i in y_pred_idx])
    return y_true, y_pred


y_true_cnn, y_pred_cnn = get_predictions(custom_cnn, test_dataset_1ch)
y_true_dense, y_pred_dense = get_predictions(densenet, test_dataset_3ch)

results_cnn = evaluate_predictions(y_true_cnn, y_pred_cnn, CLASS_NAMES)
results_dense = evaluate_predictions(y_true_dense, y_pred_dense, CLASS_NAMES)

## Summary comparison table

**Intent**: this is the table that goes into `README.md`'s Results section and the
dashboard's Model Performance page.

In [ ]:
comparison = pd.DataFrame(
    {
        "Custom CNN": [
            results_cnn["report"].loc["accuracy"].iloc[0],
            results_cnn["report"].loc["macro avg", "f1-score"],
            results_cnn["no_tumor_miss_rate"],
        ],
        "DenseNet121 (transfer)": [
            results_dense["report"].loc["accuracy"].iloc[0],
            results_dense["report"].loc["macro avg", "f1-score"],
            results_dense["no_tumor_miss_rate"],
        ],
    },
    index=["Accuracy", "Macro F1", "No-tumor miss rate"],
)

comparison

## Confusion matrices side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, results, title in zip(
    axes, [results_cnn, results_dense], ["Custom CNN", "DenseNet121 (transfer)"], strict=True
):
    cm = results["confusion_matrix"]
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_title(title)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, cm.iloc[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()

Next notebook: **07_leakage_ablation.ipynb** — the strongest evidence piece: same
DenseNet121 architecture, naive slice-level split vs. patient-level split, side by
side.